# AgenticBBO-Bench: toy replay of the agent-as-optimizer findings

Paper: "A Closer Look at Agentic BBO: Benchmarking LLM Agents for Black-Box Optimization" —
Ming Chen, Rong-Xi Tan, Ke Xue et al. (Nanjing University LAMDA), arXiv:2610.12183v1, Oct 8 2026.

This notebook does NOT reproduce the paper's runs (those used real HPO / PostgreSQL / chip-placement tasks under a Codex harness). It replays the *mechanisms* the paper's ablations isolated on a cheap multimodal black-box objective: (1) agentic > direct, (2) task semantics beat extra tools, (3) agent warm-up + numerical handoff matches continued online search. Reported paper scores are reproduced as tables in the last cell.

In [ ]:
import numpy as np
from scipy.optimize import minimize

def blackbox(x):
    """5-D multimodal objective. Semantics: the good region sits near [0.5, -1.5, 0.5, -1.5, 0.5];
    a strong-but-narrow decoy basin near [-3.5, 3.0, -3.5, 3.0, -2.5] traps cold local optimizers."""
    x = np.asarray(x, dtype=float)
    d_good = np.array([0.5, -1.5, 0.5, -1.5, 0.5])
    d_decoy = np.array([-3.5, 3.0, -3.5, 3.0, -2.5])
    v = (2.2 * np.exp(-0.55 * np.sum((x - d_decoy) ** 2))
         + 3.0 * np.exp(-0.18 * np.sum((x - d_good) ** 2))
         + 0.4 * np.sin(3.0 * x[0] + 1.7) * np.cos(2.4 * x[1] - 0.6)
         - 0.06 * np.sum(x ** 2))
    return -v  # minimize; best achievable ~ -3.1

def random_search(budget, bounds=(-5.0, 5.0), seed=0):
    """Classical dumb baseline: uniform random draws (like Sobol/Random in the paper)."""
    r = np.random.default_rng(seed)
    xs = r.uniform(bounds[0], bounds[1], size=(budget, 5))
    vals = np.array([blackbox(x) for x in xs])
    return vals.min(), xs[vals.argmin()]

def direct_guess(budget, seed=0):
    """'Direct' LLM prompting: one-shot plausible-looking guesses, no reasoning, no tools."""
    r = np.random.default_rng(seed)
    xs = r.choice([-3.0, -1.0, 0.0, 1.0, 3.0], size=(budget, 5)).astype(float)
    xs += r.normal(0, 0.25, size=(budget, 5))
    vals = np.array([blackbox(x) for x in xs])
    return vals.min(), xs[vals.argmin()]

def agentic_search(budget, seed=0):
    """Agentic loop: (1) task semantics shrink the search box to [-2,2];
    (2) explore broadly in the first half; (3) exploit top-3 region in the second half;
    (4) adapt the plan mid-run (proxy refinement, like the paper's Bigblue1 case study)."""
    r = np.random.default_rng(seed)
    half = budget // 2
    hist_x, hist_v = [], []
    for _ in range(half):
        x = r.uniform(-2.0, 2.0, size=5)
        hist_x.append(x); hist_v.append(blackbox(x))
    top3 = np.argsort(hist_v)[:3]
    anchor = np.mean([hist_x[i] for i in top3], axis=0)
    for _ in range(half, budget):
        x = anchor + r.normal(0, 0.55, size=5)
        hist_x.append(x); hist_v.append(blackbox(x))
        if hist_v[-1] < hist_v[int(top3[0])]:
            anchor = 0.7 * x + 0.3 * anchor
    i = int(np.argmin(hist_v))
    return hist_v[i], hist_x[i]

B = 40
SEEDS = range(8)
print('objective ready; budget =', B)

In [ ]:
# Core claim (paper section 5.2): agentic > direct in all five families (+30.3% avg).
r_rand = [random_search(B, seed=s)[0] for s in SEEDS]
r_dir = [direct_guess(B, seed=s)[0] for s in SEEDS]
r_agent = [agentic_search(B, seed=s)[0] for s in SEEDS]
m_rand, m_dir, m_agent = np.mean(r_rand), np.mean(r_dir), np.mean(r_agent)
print(f"mean best over 8 seeds (lower is better):")
print(f"  random={m_rand:.3f}  direct={m_dir:.3f}  agentic={m_agent:.3f}")
assert m_agent < m_rand, "agentic should beat plain random search"
assert m_agent < m_dir, "agentic should beat one-shot direct guessing"

In [ ]:
# Paper Table 2(b): extra numerical tools do not consistently help (0.589 > 0.574 > 0.538).
# Paper section 5.5: agent warm-up then hand off to a numerical optimizer matches continued online search.
def agentic_plus_gp_tools(budget, seed=0):
    """Base agentic phase, then a fixed tool: multi-start local minimization (rigid inductive bias)."""
    v, x0 = agentic_search(budget - 6, seed=seed)
    best = v
    for s in range(6):
        res = minimize(blackbox, x0 + np.random.default_rng(seed + s).normal(0, 0.1, 5),
                       method='L-BFGS-B', bounds=[(-5, 5)] * 5)
        best = min(best, res.fun)
    return best

def agent_then_numerical_handoff(budget, seed=0):
    """Agent explores the first half, then a numerical optimizer takes over from its best region."""
    _, x0 = agentic_search(budget // 2, seed=seed)
    res = minimize(blackbox, x0, method='L-BFGS-B', bounds=[(-5, 5)] * 5)
    return res.fun

def pure_numerical(budget, seed=0):
    """Same numerical optimizer, cold start — no agent warm-up (like TuRBO from scratch)."""
    r = np.random.default_rng(seed)
    res = minimize(blackbox, r.uniform(-5, 5, 5), method='L-BFGS-B', bounds=[(-5, 5)] * 5)
    return res.fun

h_agent = [agentic_search(B, seed=s)[0] for s in SEEDS]
h_gp = [agentic_plus_gp_tools(B, seed=s) for s in SEEDS]
h_hand = [agent_then_numerical_handoff(B, seed=s) for s in SEEDS]
h_cold = [pure_numerical(B, seed=s) for s in SEEDS]
print(f"base agentic={np.mean(h_agent):.3f}  agentic+fixed-tools={np.mean(h_gp):.3f}")
print(f"handoff={np.mean(h_hand):.3f}  numerical-cold={np.mean(h_cold):.3f}")
assert np.mean(h_hand) <= np.mean(h_cold), "warm-started numerical should beat cold-started numerical"
# paper: agent->TuRBO handoff 0.570 ~= online agentic 0.569 >> TuRBO cold 0.475

In [ ]:
# Paper section 5.4: semantics 0.465 -> 0.595; full prior unreliable 0.569; wrong support kills the gain.
def agentic_anonymous(budget, seed=0):
    """Agent sees anonymous knobs: full-space exploration, no semantic box-shrink."""
    r = np.random.default_rng(seed)
    hist_x, hist_v = [], []
    for _ in range(budget // 2):
        x = r.uniform(-5.0, 5.0, size=5)
        hist_x.append(x); hist_v.append(blackbox(x))
    top3 = np.argsort(hist_v)[:3]
    anchor = np.mean([hist_x[i] for i in top3], axis=0)
    for _ in range(budget // 2, budget):
        x = anchor + r.normal(0, 0.55, size=5)
        hist_x.append(x); hist_v.append(blackbox(x))
    return min(hist_v)

def agentic_wrong_prior(budget, seed=0):
    """Agent given a confident-but-wrong prior: search concentrated near the decoy basin."""
    r = np.random.default_rng(seed)
    best = np.inf
    for _ in range(budget):
        x = np.array([-3.5, 3.0, -3.5, 3.0, -2.5]) + r.normal(0, 0.8, size=5)
        best = min(best, blackbox(x))
    return best

s_anon = [agentic_anonymous(B, seed=s) for s in SEEDS]
s_sem = [agentic_search(B, seed=s)[0] for s in SEEDS]
s_wrong = [agentic_wrong_prior(B, seed=s) for s in SEEDS]
print(f"anonymous={np.mean(s_anon):.3f}  semantic={np.mean(s_sem):.3f}  wrong-prior={np.mean(s_wrong):.3f}")
assert np.mean(s_sem) < np.mean(s_anon), "task semantics should beat anonymous knobs"
assert np.mean(s_wrong) > np.mean(s_sem), "wrong prior should wipe out the gain (paper: 0.872 -> 0.682)"

In [ ]:
# Reference: the paper's reported numbers (higher = better, normalized scores).
print("AgenticBBO-Bench main results (Table 2a), normalized score, higher is better:")
print("  family    Direct  Agentic  best numerical")
rows = [("BBOB", 0.460, 0.661, 0.592), ("HPO", 0.519, 0.554, 0.432),
        ("DBTune", 0.388, 0.525, 0.660), ("BBOPlace", 0.465, 0.544, 0.527),
        ("GuacaMol", 0.565, 0.840, 0.127)]
for fam, d, a, n in rows:
    print(f"  {fam:8s}  {d:.3f}   {a:.3f}    {n:.3f}")
    assert a > d
assert sum(1 for _, d, a, n in rows if a > n) == 4, "agentic best in 4/5 families"
print("\n+30.3% avg relative improvement agentic-over-direct (BBOB +43.7%, GuacaMol +48.7%).")
print("Frontier challenge (7 LLMs, Codex harness): GPT-6 Astra 0.591, DeepSeek V4.1 Flash 0.584 —")
print("both on the performance-vs-cost Pareto frontier; score only weakly aligned with token cost.")
print("Tool ablation means: base 0.589 > +GP Suggest 0.574 > +GP Tools 0.538.")
print("Semantics: 0.465 (anonymous) -> 0.595 (semantic) -> 0.569 (full prior, unreliable).")
print("Role: agent->TuRBO handoff 0.570 ~= online agentic 0.569 >> TuRBO cold 0.475.")